In [1]:
"""
Construye un dataset reducido para descarga, a partir de df_completo_view.csv:

  - Imágenes con ViewPosition en {AP, PA}
  - Más TODAS las bad-quality (calidad-imagen == 0), aunque no sean AP/PA
  - Schema compatible con df_subset_pa_ap.csv + columna ViewPosition extra
  - El full_path resultante apunta al destino LOCAL en Windows, no a PhysioNet

Salida: df_subset_reducido.csv en la misma carpeta que el input. Después se
alimenta a download_bad_quality.py para bajar las imágenes.
"""

from pathlib import Path
import pandas as pd


# ---- Config ----------------------------------------------------------------
INPUT_PATH = Path(
    r"C:\Users\trodr\Documents\proyecto-torax-v2.0"
    r"\01-dataset\dataset-completo-merge\df_completo_view.csv"
)
OUTPUT_PATH = INPUT_PATH.with_name("df_subset_reducido.csv")

# Carpeta local donde se guardarán las imágenes descargadas.
# La estructura interna será: <root>\s<study_id>\<dicom_id>.jpg
LOCAL_TARGET_ROOT = (
    r"C:\Users\trodr\Documents\proyecto-torax-v2.0"
    r"\01-dataset\dataset_prueba\subset_reducido"
)

# Si querés muestrear las AP/PA, poné un entero. Las bad-quality SIEMPRE se
# incluyen completas — no las toca el sampling.
#   None  -> todas las AP/PA (~243k filas, ~80-100 GB de descarga)
#   5000  -> piloto razonable
#   2000  -> piloto chico
SAMPLE_N = None
RANDOM_SEED = 42

# Mismo schema que df_subset_pa_ap.csv (18 cols) + ViewPosition.
LABEL_COLS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Enlarged Cardiomediastinum", "Fracture", "Lung Lesion", "Lung Opacity",
    "No Finding", "Pleural Effusion", "Pleural Other", "Pneumonia",
    "Pneumothorax", "Support Devices",
]
# ---------------------------------------------------------------------------


def main() -> None:
    print(f"Leyendo: {INPUT_PATH}")
    df = pd.read_csv(INPUT_PATH)
    print(f"  shape: {df.shape}")
    print()

    # --- Máscaras ---------------------------------------------------------
    is_ap_pa = df["ViewPosition"].isin(["AP", "PA"])
    is_bad = df["calidad-imagen"] == 0.0

    print(f"AP/PA totales:          {is_ap_pa.sum()}")
    print(f"Bad-quality totales:    {is_bad.sum()}")
    print(f"Overlap AP/PA & Bad:    {(is_ap_pa & is_bad).sum()}")
    print(f"Bad-quality NO AP/PA:   {(is_bad & ~is_ap_pa).sum()}  (se incluyen igual)")

    # --- Sampling de AP/PA, si aplica ------------------------------------
    ap_pa_set = df[is_ap_pa].copy()
    if SAMPLE_N is not None and SAMPLE_N < len(ap_pa_set):
        ap_pa_set = ap_pa_set.sample(n=SAMPLE_N, random_state=RANDOM_SEED)
        print(f"\nSAMPLE_N={SAMPLE_N} -> AP/PA muestreado a {len(ap_pa_set)} filas")
    bad_set = df[is_bad].copy()

    # --- Unión + dedup por dicom_id (overlap AP/PA ∩ Bad) ----------------
    combined = pd.concat([ap_pa_set, bad_set], ignore_index=True)
    combined = combined.drop_duplicates(subset=["dicom_id"], keep="first")
    print(f"\nDataset reducido final: {len(combined)} filas")

    # --- full_path local para descarga -----------------------------------
    combined["full_path"] = combined.apply(
        lambda r: f"{LOCAL_TARGET_ROOT}\\s{r['study_id']}\\{r['dicom_id']}.jpg",
        axis=1,
    )

    # --- Schema final ----------------------------------------------------
    output_cols = (
        ["dicom_id", "subject_id", "study_id"]
        + LABEL_COLS
        + ["full_path", "ViewPosition"]
    )
    combined = combined[output_cols]

    combined.to_csv(OUTPUT_PATH, index=False)
    print(f"\nGuardado: {OUTPUT_PATH}")
    print(f"  shape: {combined.shape}")

    # --- Estimación de tamaño de descarga --------------------------------
    avg_kb_per_img = 400  # promedio razonable para MIMIC-CXR-JPG
    est_mb = len(combined) * avg_kb_per_img / 1024
    if est_mb > 1024:
        print(f"  Tamaño estimado de descarga: ~{est_mb/1024:.1f} GB")
    else:
        print(f"  Tamaño estimado de descarga: ~{est_mb:.0f} MB")

    print("\nViewPosition en el reducido:")
    print(combined["ViewPosition"].value_counts(dropna=False).to_string())


if __name__ == "__main__":
    main()

Leyendo: C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\df_completo_view.csv
  shape: (377110, 25)

AP/PA totales:          243334
Bad-quality totales:    55
Overlap AP/PA & Bad:    30
Bad-quality NO AP/PA:   25  (se incluyen igual)

Dataset reducido final: 243359 filas

Guardado: C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\df_subset_reducido.csv
  shape: (243359, 19)
  Tamaño estimado de descarga: ~92.8 GB

ViewPosition en el reducido:
ViewPosition
AP         147173
PA          96161
LL             22
NaN             2
LATERAL         1


In [2]:
"""
Genera un CSV de descarga con:
  - 500 imágenes AP/PA sampleadas (random con semilla fija)
  - + las 30 imágenes bad-quality AP/PA forzadas
  - Dedup por dicom_id -> ~530 filas únicas

Input:  df_completo_view.csv  (necesita ViewPosition + calidad-imagen)
Output: df_subset_500.csv     (mismo schema que df_subset_pa_ap.csv + ViewPosition)
"""

from pathlib import Path
import pandas as pd


# ---- Config ----------------------------------------------------------------
INPUT_PATH = Path(
    r"C:\Users\trodr\Documents\proyecto-torax-v2.0"
    r"\01-dataset\dataset-completo-merge\df_completo_view.csv"
)
OUTPUT_PATH = INPUT_PATH.with_name("df_subset_500.csv")

# Carpeta local donde se guardarán las imágenes al descargarlas.
# Estructura interna: <root>\s<study_id>\<dicom_id>.jpg
LOCAL_TARGET_ROOT = (
    r"C:\Users\trodr\Documents\proyecto-torax-v2.0"
    r"\01-dataset\dataset_prueba\subset_500"
)

SAMPLE_N = 500
RANDOM_SEED = 42

LABEL_COLS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Enlarged Cardiomediastinum", "Fracture", "Lung Lesion", "Lung Opacity",
    "No Finding", "Pleural Effusion", "Pleural Other", "Pneumonia",
    "Pneumothorax", "Support Devices",
]
# ---------------------------------------------------------------------------


def main() -> None:
    print(f"Leyendo: {INPUT_PATH}")
    df = pd.read_csv(INPUT_PATH)
    print(f"  shape: {df.shape}")

    # --- Máscaras ---------------------------------------------------------
    is_ap_pa = df["ViewPosition"].isin(["AP", "PA"])
    is_bad_ap_pa = is_ap_pa & (df["calidad-imagen"] == 0.0)
    is_good_ap_pa = is_ap_pa & (df["calidad-imagen"] != 0.0)

    n_ap_pa = is_ap_pa.sum()
    n_bad_ap_pa = is_bad_ap_pa.sum()
    print(f"\nAP/PA totales:      {n_ap_pa}")
    print(f"Bad AP/PA totales:  {n_bad_ap_pa}")

    # --- Sample de AP/PA (excluyendo bad para evitar overlap aleatorio) ---
    # Sampleamos del pool de buenas (o NaN), después forzamos las bad
    # encima. Asegura que las bad SIEMPRE estén y que el sample tenga
    # exactamente SAMPLE_N buenas/NaN, sin perderse por colisión.
    sample = df[is_good_ap_pa].sample(n=SAMPLE_N, random_state=RANDOM_SEED)
    bad = df[is_bad_ap_pa]
    combined = pd.concat([sample, bad], ignore_index=True)

    # Sanity check: dedup defensivo (no debería haber duplicados acá)
    before = len(combined)
    combined = combined.drop_duplicates(subset=["dicom_id"], keep="first")
    if len(combined) != before:
        print(f"  Aviso: {before - len(combined)} duplicados removidos")

    print(f"\nDataset final: {len(combined)} filas")
    print(f"  De los cuales bad AP/PA: {(combined['calidad-imagen'] == 0.0).sum()}")

    # --- full_path local -------------------------------------------------
    combined["full_path"] = combined.apply(
        lambda r: f"{LOCAL_TARGET_ROOT}\\s{r['study_id']}\\{r['dicom_id']}.jpg",
        axis=1,
    )

    # --- Schema de salida ------------------------------------------------
    output_cols = (
        ["dicom_id", "subject_id", "study_id"]
        + LABEL_COLS
        + ["full_path", "ViewPosition"]
    )
    missing = [c for c in output_cols if c not in combined.columns]
    if missing:
        raise SystemExit(f"ERROR: faltan columnas en el input: {missing}")
    combined = combined[output_cols]

    combined.to_csv(OUTPUT_PATH, index=False)
    print(f"\nGuardado: {OUTPUT_PATH}")
    print(f"  shape: {combined.shape}")
    print(f"\nViewPosition distribución:")
    print(combined["ViewPosition"].value_counts().to_string())


if __name__ == "__main__":
    main()

Leyendo: C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\df_completo_view.csv
  shape: (377110, 25)

AP/PA totales:      243334
Bad AP/PA totales:  30

Dataset final: 530 filas
  De los cuales bad AP/PA: 30

Guardado: C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\df_subset_500.csv
  shape: (530, 19)

ViewPosition distribución:
ViewPosition
AP    333
PA    197


In [6]:
df = pd.read_csv(r'C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\df_subset_500.csv')

In [8]:
df.head(5)

,dicom_id,subject_id,study_id,Atelectasis,Cardiomegaly,Consolidation,Edema,Enlarged Cardiomediastinum,Fracture,Lung Lesion,Lung Opacity,No Finding,Pleural Effusion,Pleural Other,Pneumonia,Pneumothorax,Support Devices,full_path,ViewPosition,calidad-imagen
0,1621b732-f77b9463-87769d53-3175e5ed-1822d35f,10516278,53994171,NaN,NaN,NaN,0.0,NaN,NaN,NaN,1.0,NaN,0.0,NaN,0.0,NaN,NaN,C:\Users\trodr\Documents\proyecto-torax-v2.0\0...,AP,NaN
1,a9dfa70d-5d8b5850-55eb30a9-bbceba45-50d136a6,14462350,57896727,1.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,C:\Users\trodr\Documents\proyecto-torax-v2.0\0...,AP,1.0
2,9fcb75dd-5d039dc4-83cb6de7-0ebcdc22-1af9fcef,19431075,59114744,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,1.0,NaN,NaN,C:\Users\trodr\Documents\proyecto-torax-v2.0\0...,AP,1.0
3,a4a85001-3068f851-a1baae43-868d1727-1bf840dc,16129520,57623500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,C:\Users\trodr\Documents\proyecto-torax-v2.0\0...,PA,NaN
4,1c4bcdef-37e2de53-a6fe1089-216fcceb-2ac01aa6,15447983,58598537,1.0,-1.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,C:\Users\trodr\Documents\proyecto-torax-v2.0\0...,AP,NaN


In [5]:
"""
Genera un CSV de descarga con:
  - 500 imágenes AP/PA sampleadas (random con semilla fija)
  - + las 30 imágenes bad-quality AP/PA forzadas
  - Dedup por dicom_id -> ~530 filas únicas

Input:  df_completo_view.csv  (necesita ViewPosition + calidad-imagen)
Output: df_subset_500.csv     (mismo schema que df_subset_pa_ap.csv + ViewPosition)
"""

from pathlib import Path
import pandas as pd


# ---- Config ----------------------------------------------------------------
INPUT_PATH = Path(
    r"C:\Users\trodr\Documents\proyecto-torax-v2.0"
    r"\01-dataset\dataset-completo-merge\df_completo_view.csv"
)
OUTPUT_PATH = INPUT_PATH.with_name("df_subset_500.csv")

# Carpeta local donde se guardarán las imágenes al descargarlas.
# Estructura interna: <root>\s<study_id>\<dicom_id>.jpg
LOCAL_TARGET_ROOT = (
    r"C:\Users\trodr\Documents\proyecto-torax-v2.0"
    r"\01-dataset\dataset_prueba\subset_500"
)

SAMPLE_N = 500
RANDOM_SEED = 42

LABEL_COLS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Enlarged Cardiomediastinum", "Fracture", "Lung Lesion", "Lung Opacity",
    "No Finding", "Pleural Effusion", "Pleural Other", "Pneumonia",
    "Pneumothorax", "Support Devices",
]
# ---------------------------------------------------------------------------


def main() -> None:
    print(f"Leyendo: {INPUT_PATH}")
    df = pd.read_csv(INPUT_PATH)
    print(f"  shape: {df.shape}")

    # --- Máscaras ---------------------------------------------------------
    is_ap_pa = df["ViewPosition"].isin(["AP", "PA"])
    is_bad_ap_pa = is_ap_pa & (df["calidad-imagen"] == 0.0)
    is_good_ap_pa = is_ap_pa & (df["calidad-imagen"] != 0.0)

    n_ap_pa = is_ap_pa.sum()
    n_bad_ap_pa = is_bad_ap_pa.sum()
    print(f"\nAP/PA totales:      {n_ap_pa}")
    print(f"Bad AP/PA totales:  {n_bad_ap_pa}")

    # --- Sample de AP/PA (excluyendo bad para evitar overlap aleatorio) ---
    # Sampleamos del pool de buenas (o NaN), después forzamos las bad
    # encima. Asegura que las bad SIEMPRE estén y que el sample tenga
    # exactamente SAMPLE_N buenas/NaN, sin perderse por colisión.
    sample = df[is_good_ap_pa].sample(n=SAMPLE_N, random_state=RANDOM_SEED)
    bad = df[is_bad_ap_pa]
    combined = pd.concat([sample, bad], ignore_index=True)

    # Sanity check: dedup defensivo (no debería haber duplicados acá)
    before = len(combined)
    combined = combined.drop_duplicates(subset=["dicom_id"], keep="first")
    if len(combined) != before:
        print(f"  Aviso: {before - len(combined)} duplicados removidos")

    print(f"\nDataset final: {len(combined)} filas")
    print(f"  De los cuales bad AP/PA: {(combined['calidad-imagen'] == 0.0).sum()}")

    # --- full_path local -------------------------------------------------
    combined["full_path"] = combined.apply(
        lambda r: f"{LOCAL_TARGET_ROOT}\\s{r['study_id']}\\{r['dicom_id']}.jpg",
        axis=1,
    )

    # --- Schema de salida ------------------------------------------------
    output_cols = (
        ["dicom_id", "subject_id", "study_id"]
        + LABEL_COLS
        + ["full_path", "ViewPosition", "calidad-imagen"]
    )
    missing = [c for c in output_cols if c not in combined.columns]
    if missing:
        raise SystemExit(f"ERROR: faltan columnas en el input: {missing}")
    combined = combined[output_cols]

    combined.to_csv(OUTPUT_PATH, index=False)
    print(f"\nGuardado: {OUTPUT_PATH}")
    print(f"  shape: {combined.shape}")
    print(f"\nViewPosition distribución:")
    print(combined["ViewPosition"].value_counts().to_string())


if __name__ == "__main__":
    main()

Leyendo: C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\df_completo_view.csv
  shape: (377110, 25)

AP/PA totales:      243334
Bad AP/PA totales:  30

Dataset final: 530 filas
  De los cuales bad AP/PA: 30

Guardado: C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\df_subset_500.csv
  shape: (530, 20)

ViewPosition distribución:
ViewPosition
AP    333
PA    197


In [9]:
"""
Genera un .txt con las rutas relativas en formato PhysioNet a partir de
df_subset_500.csv, listo para usar con wget -i.

Formato por línea:
    files/pXX/p<subject_id>/s<study_id>/<dicom_id>.jpg

Con CRLF (Windows) para mantener compatibilidad con los otros .txt del proyecto.
"""

from pathlib import Path
import pandas as pd


# ---- Config ----------------------------------------------------------------
INPUT_CSV = Path(
    r"C:\Users\trodr\Documents\proyecto-torax-v2.0"
    r"\01-dataset\dataset-completo-merge\df_subset_500.csv"
)
OUTPUT_TXT = INPUT_CSV.with_name("subset500_Calidad.txt")
# ---------------------------------------------------------------------------


def build_physionet_path(row) -> str:
    """files/pXX/p<subject>/s<study>/<dicom>.jpg"""
    subject_id = str(int(row["subject_id"]))
    study_id = str(int(row["study_id"]))
    prefix = subject_id[:2]   # ej 10003400 -> "10"
    return f"files/p{prefix}/p{subject_id}/s{study_id}/{row['dicom_id']}.jpg"


def main() -> None:
    print(f"Leyendo: {INPUT_CSV}")
    df = pd.read_csv(INPUT_CSV)
    print(f"  shape: {df.shape}")

    required = {"dicom_id", "subject_id", "study_id"}
    missing = required - set(df.columns)
    if missing:
        raise SystemExit(f"ERROR: faltan columnas {missing} en el CSV")

    paths = df.apply(build_physionet_path, axis=1).tolist()

    # CRLF para que coincida con MI_SUBSET_PA_AP.txt
    with open(OUTPUT_TXT, "w", newline="") as f:
        for p in paths:
            f.write(p + "\r\n")

    print(f"\nGuardado: {OUTPUT_TXT}")
    print(f"  líneas:   {len(paths)}")
    print(f"\nPrimeras 3 líneas:")
    for p in paths[:3]:
        print(f"  {p}")


if __name__ == "__main__":
    main()

Leyendo: C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\df_subset_500.csv
  shape: (530, 20)

Guardado: C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\subset500_Calidad.txt
  líneas:   530

Primeras 3 líneas:
  files/p10/p10516278/s53994171/1621b732-f77b9463-87769d53-3175e5ed-1822d35f.jpg
  files/p14/p14462350/s57896727/a9dfa70d-5d8b5850-55eb30a9-bbceba45-50d136a6.jpg
  files/p19/p19431075/s59114744/9fcb75dd-5d039dc4-83cb6de7-0ebcdc22-1af9fcef.jpg
